In [2]:

import numpy as np
import pandas as pd
import joblib

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score
)

from imblearn.over_sampling import SMOTENC
from imblearn.pipeline import Pipeline


# 1. Generate synthetic transaction data
def generate_demo_data(n_samples=20000, fraud_rate=0.02, random_state=42):
    if not 0 < fraud_rate < 1:
        raise ValueError("fraud_rate must be between 0 and 1.")

    rng = np.random.default_rng(random_state)

    # Transaction amount
    amounts = np.round(
        rng.exponential(scale=50, size=n_samples), 2
    )

    # Seconds since midnight
    times = rng.integers(0, 86400, size=n_samples)

    # Transaction types
    transaction_types = rng.choice(
        ["ATM", "POS", "Online"],
        size=n_samples,
        p=[0.40, 0.45, 0.15]
    )

    # Transaction locations
    locations = rng.choice(
        ["Abuja", "Lagos", "Kano", "Others"],
        size=n_samples,
        p=[0.25, 0.45, 0.15, 0.15]
    )

    # Risk score: larger scores increase fraud probability
    risk_score = np.ones(n_samples)

    risk_score += (amounts > 200) * 4
    risk_score += (transaction_types == "Online") * 6
    risk_score += (
        (times < 4 * 3600) | (times > 20 * 3600)
    ) * 2
    risk_score += (locations == "Others") * 1

    # Convert risk scores into probabilities.
    # fraud_rate is the approximate baseline prevalence.
    probs = fraud_rate * risk_score
    probs = np.clip(probs, 0, 0.95)

    labels = (rng.random(n_samples) < probs).astype(int)

    df = pd.DataFrame({
        "amount": amounts,
        "time": times,
        "transaction_type": transaction_types,
        "location": locations,
        "class": labels
    })

    return df


df = generate_demo_data(n_samples=20000)

print("Dataset shape:", df.shape)
print("\nClass distribution:")
print(df["class"].value_counts())
print("\nClass proportions:")
print(df["class"].value_counts(normalize=True))


# 2. Encode categorical features consistently
transaction_map = {
    "ATM": 0,
    "POS": 1,
    "Online": 2
}

location_map = {
    "Abuja": 0,
    "Lagos": 1,
    "Kano": 2,
    "Others": 3
}

df["transaction_type"] = df["transaction_type"].map(
    transaction_map
)

df["location_code"] = df["location"].map(location_map)


# 3. Prepare features (X) and target (y)
feature_columns = [
    "amount",
    "time",
    "transaction_type",
    "location_code"
]

X = df[feature_columns]
y = df["class"]


# 4. Split data before resampling
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    stratify=y,
    random_state=42
)

print("\nTraining class distribution:")
print(y_train.value_counts())

print("\nTesting class distribution:")
print(y_test.value_counts())


# 5. Preprocess numerical columns and preserve categorical columns
preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            StandardScaler(),
            ["amount", "time"]
        ),
        (
            "categorical",
            "passthrough",
            ["transaction_type", "location_code"]
        )
    ],
    verbose_feature_names_out=False
)


# 6. Build the pipeline
# Column order after preprocessing:
# 0 = amount, 1 = time, 2 = transaction_type, 3 = location_code

pipeline = Pipeline([
    ("preprocessor", preprocessor),

    ("smote", SMOTENC(
        categorical_features=[2, 3],
        random_state=42
    )),

    ("classifier", RandomForestClassifier(
        n_estimators=150,
        random_state=42,
        n_jobs=-1
    ))
])


# 7. Train the model
pipeline.fit(X_train, y_train)

print("\nModel training completed.")


# 8. Evaluate the model on the original, untouched test data
y_pred = pipeline.predict(X_test)
y_proba = pipeline.predict_proba(X_test)[:, 1]

print("\nClassification report:")
print(classification_report(
    y_test,
    y_pred,
    labels=[0, 1],
    target_names=["Legitimate", "Fraud"],
    zero_division=0
))

print("Confusion matrix:")
print(confusion_matrix(y_test, y_pred, labels=[0, 1]))

if y_test.nunique() == 2:
    print("ROC AUC:", roc_auc_score(y_test, y_proba))
else:
    print("ROC AUC cannot be calculated: test data has one class.")


# 9. Save the complete trained pipeline
model_path = "fraud_model.pkl"
joblib.dump(pipeline, model_path)

print(f"\nSaved model to {model_path}")


# 10. Load the model and predict a sample transaction
model = joblib.load(model_path)

sample = pd.DataFrame([{
    "amount": 5000.0,
    "time": 45000,
    "transaction_type": transaction_map["POS"],
    "location_code": location_map["Lagos"]
}])

prediction = model.predict(sample)[0]
fraud_probability = model.predict_proba(sample)[0, 1]

print("\nSample transaction:")
print(sample)

print(
    "Prediction:",
    "Fraud" if prediction == 1 else "Legitimate"
)

print(f"Estimated fraud probability: {fraud_probability:.2%}")


Dataset shape: (20000, 5)

Class distribution:
class
0    18913
1     1087
Name: count, dtype: int64

Class proportions:
class
0    0.94565
1    0.05435
Name: proportion, dtype: float64

Training class distribution:
class
0    15130
1      870
Name: count, dtype: int64

Testing class distribution:
class
0    3783
1     217
Name: count, dtype: int64

Model training completed.

Classification report:
              precision    recall  f1-score   support

  Legitimate       0.95      0.78      0.86      3783
       Fraud       0.07      0.30      0.12       217

    accuracy                           0.76      4000
   macro avg       0.51      0.54      0.49      4000
weighted avg       0.90      0.76      0.82      4000

Confusion matrix:
[[2960  823]
 [ 152   65]]
ROC AUC: 0.6086414970685007

Saved model to fraud_model.pkl

Sample transaction:
   amount   time  transaction_type  location_code
0  5000.0  45000                 1              1
Prediction: Legitimate
Estimated fraud probab